# 멀티모달 LLM - 실습 코드 3: GPT-4V 멀티모달 API 사용

- Tutorial ID: `expand-multimodal-llm`
- Tutorial: 멀티모달 LLM
- Section ID: `expand-multimodal-llm-code-3`
- Section: 실습 코드 3: GPT-4V 멀티모달 API 사용


In [ ]:
# ============================================================
# 코드 읽는 법 — 실습 코드 3: GPT-4V 멀티모달 API 사용
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) Q/K/V가 어떤 shape으로 만들어지고 attention score로 이어지는지 추적
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
from openai import OpenAI
import base64

client = OpenAI()

def encode_image(image_path):
    """이미지를 Base64로 인코딩"""
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

# ── 이미지 분석 (GPT-4V) ──
def analyze_image(image_path, question):
    """GPT-4V로 이미지 분석"""
    base64_image = encode_image(image_path)
    
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": question},
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/jpeg;base64,{base64_image}",
                            "detail": "high"  # 고해상도 분석
                        }
                    }
                ]
            }
        ],
        max_tokens=500,
    )
    return response.choices[0].message.content

# ── 이미지 URL로 직접 분석 ──
def analyze_image_url(image_url, question):
    """URL 이미지 분석"""
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": question},
                    {"type": "image_url", "image_url": {"url": image_url}}
                ]
            }
        ],
    )
    return response.choices[0].message.content

# 다양한 분석 태스크
tasks = [
    "이 이미지를 상세히 묘사하세요.",
    "이미지에 있는 텍스트를 모두 읽어세요 (OCR).",
    "이 이미지에서 이상한 점이 있나요?",
    "이 이미지로 요리 레시피를 추론하세요.",
]

image_url = "http://images.cocodataset.org/val2017/000000039769.jpg"
print("=== GPT-4V 멀티모달 분석 ===\n")
for task in tasks:
        result = analyze_image_url(image_url, task)
    print(f"질문: {task}")
    print(f"답변: {result[:200]}...\n")